# Continuous Distribution Dequantization for Observational Data

Observational meteorological datasets frequently suffer from **discretization and reporting resolution artifacts** (e.g., temperature rounded to integers). These discrete "teeth" impair non-parametric distribution fitting, quantile mapping, and extreme event analysis.

This notebook applies an adaptive **Smoothed Empirical Cumulative Distribution Function (Smoothed ECDF)** framework to restore a continuous, smooth probability density function while strictly preserving long-term active physical moments (mean, sample variance) and masking missing observational values.

## Step 1: Import Dependencies

In [ ]:
import climakitae as ck
import xarray as xr
import pandas as pd
import geopandas as gpd
import intake
import hvplot.xarray
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from xclim.core.units import convert_units_to
import scipy.stats as stats
import scipy.signal as signal
from sklearn.neighbors import KernelDensity

# Global Plotting Setup
plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (11, 5)

## Step 2: Methodological Foundations & Functional Definitions

1. **Preprocessing:** Observations for all variables are _adaptively_ dithered.
    - **Justification for dithering:** Discrete reporting resolutions (e.g., temperatures recorded to the nearest quarter-degree) create artificial probability mass concentrations. Dithering injects controlled uniform noise bounded by half the localized reporting step ($\pm \Delta / 2$). This un-collects discrete "teeth" back into continuous interval distributions without shifting variable means or distorting real variance boundaries.
    - **Definition of _adaptive_**:
        - A given variable's observations contain inconsistent reporting steps (See Step 4). For instance, `tas` has larger gaps at the tails of the distributions due to the sparsity of data there; meanwhile, the steps at the dense core of the distributions are much smaller.
        - The adaptive dithering step finds the median reporting interval through the distribution to determine the largest possible $\Delta$ between observations.
        - If two observations have a smaller $\Delta$ between them, a local $\Delta$ is computed to ensure there is no bleeding across multiple neighboring observations.
        - The dithering between each observation is constrained by $[-\Delta/2, \Delta/2]$, where the $\Delta$ is either the median or local interval as appropriate.
    

2. **Smoothing Temperature (`tas`, `tdps`) — Unconstrained Domain:** Fit a standard Gaussian KDE; `tas` follows a classic Gaussian distribution;`tdps` is a skewed Gaussian.

In [ ]:
# =====================================================================
# BASIC QA FUNCTIONS
# =====================================================================

def apply_qaqc_and_plot_missing(
    obs_ds: xr.Dataset,
    stn_name: str,
    qaqc_suffix: str = "_eraqc",
) -> xr.Dataset:
    """Apply quality control flags to mask invalid observations and plot temporal patterns.

    Parameters
    ----------
    obs_ds : xr.Dataset
        Raw input dataset containing meteorological variables and QAQC flags.
    stn_name : str
        Station identifier name for figure titling.
    qaqc_suffix : str, default="_eraqc"
        Suffix identifying quality control flag variables.

    Returns
    -------
    xr.Dataset
        Cleaned Dataset with flagged observations set to NaN.
    """
    clean_ds = xr.Dataset(attrs=obs_ds.attrs)
    qa_vars = [var for var in obs_ds.data_vars if var.endswith(qaqc_suffix)]
    base_vars = [
        var.replace(qaqc_suffix, "")
        for var in qa_vars
        if var.replace(qaqc_suffix, "") in obs_ds
    ]

    # Mask base variable data where QAQC flag condition is populated
    for var in base_vars:
        qa_var = f"{var}{qaqc_suffix}"
        condition = obs_ds[qa_var].notnull()
        clean_ds[var] = obs_ds[var].where(~condition, np.nan)

    # Aggregate missingness percentage across temporal dimensions
    missing_pct = {}
    for var in base_vars:
        is_null = clean_ds[var].isnull()
        by_year = is_null.groupby("time.year").mean(dim=xr.ALL_DIMS) * 100
        by_month = is_null.groupby("time.month").mean(dim=xr.ALL_DIMS) * 100
        missing_pct[var] = {"year": by_year, "month": by_month}

    # Construct summary barplots
    years = missing_pct[base_vars[0]]["year"].year.values
    months = missing_pct[base_vars[0]]["month"].month.values

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    n_vars = len(base_vars)
    bar_width = 0.8 / n_vars

    # Year plot
    x_years = np.arange(len(years))
    for i, var in enumerate(base_vars):
        offset = (i - n_vars / 2 + 0.5) * bar_width
        axes[0].bar(
            x_years + offset,
            missing_pct[var]["year"].values,
            width=bar_width,
            label=var,
        )

    step = 2 if len(years) > 20 else 1
    axes[0].set_xticks(x_years[::step])
    axes[0].set_xticklabels(years[::step], rotation=45, ha="right")
    axes[0].set_title(f"Missing Data by Year ({stn_name})")
    axes[0].set_ylabel("Missing Data (%)")
    axes[0].legend()

    # Month plot
    x_months = np.arange(len(months))
    for i, var in enumerate(base_vars):
        offset = (i - n_vars / 2 + 0.5) * bar_width
        axes[1].bar(
            x_months + offset,
            missing_pct[var]["month"].values,
            width=bar_width,
            label=var,
        )
    axes[1].set_xticks(x_months)
    axes[1].set_xticklabels(months)
    axes[1].set_title("Missing Data by Month (Totaled Across Years)")
    axes[1].set_ylabel("Missing Data (%)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    return clean_ds

def get_sorted_unique(
    ds: xr.Dataset, var_name: str, decimals: int = 3
) -> pd.DataFrame:
    """Extract, round, and count unique non-NaN values for a given variable in an xarray Dataset.

    Parameters
    ----------
    ds : xr.Dataset
        Input dataset.
    var_name : str
        Name of the target variable.
    decimals : int, default=3
        Rounding precision for unique value evaluation.

    Returns
    -------
    pd.DataFrame
        DataFrame containing sorted unique values and their frequencies.
    """
    values = ds[var_name].values.ravel()
    values = values[~np.isnan(values)]

    rounded_values = np.round(values, decimals=decimals)
    unique_vals, counts = np.unique(rounded_values, return_counts=True)

    return pd.DataFrame({"value": unique_vals, "count": counts})


# =====================================================================
# INTERNAL SMOOTHING HELPER FUNCTIONS
# =====================================================================


def _compute_adaptive_jitter(
    valid_vals: np.ndarray,
    seed: int = 902,
    cap_dithering: bool = True,
) -> tuple[np.ndarray, float]:
    """Compute local discretization steps and apply uniform dither noise."""
    rng = np.random.default_rng(seed)

    unique_vals, unique_indices = np.unique(valid_vals, return_inverse=True)

    if len(unique_vals) > 1:
        diffs = np.diff(unique_vals)
        positive_diffs = diffs[diffs > 1e-5]

        dense_core_step = (
            np.median(positive_diffs) if len(positive_diffs) > 0 else 0.1
        )

        left_diffs = np.concatenate([[np.inf], diffs])
        right_diffs = np.concatenate([diffs, [np.inf]])

        raw_half_gaps = 0.5 * np.minimum(left_diffs, right_diffs)
        raw_half_gaps[0] = 0.5 * right_diffs[0]
        raw_half_gaps[-1] = 0.5 * left_diffs[-1]

        if cap_dithering:
            half_gaps = np.minimum(raw_half_gaps, 0.5 * dense_core_step)
        else:
            half_gaps = raw_half_gaps

        local_half_gaps = half_gaps[unique_indices]
        effective_step = dense_core_step
    else:
        local_half_gaps = np.full_like(valid_vals, 0.05)
        effective_step = 0.1

    jittered = valid_vals + rng.uniform(
        -local_half_gaps, local_half_gaps, size=len(valid_vals)
    )
    return jittered, effective_step


def _fit_kde_and_map_quantiles(
    work_vals: np.ndarray,
    effective_bw: float,
    num_points: int = 5000,
) -> np.ndarray:
    """Fit Kernel Density Estimation model and perform ECDF quantile mapping."""
    n_active = len(work_vals)
    fit_data = work_vals[:, np.newaxis]

    kde = KernelDensity(kernel="gaussian", bandwidth=effective_bw).fit(fit_data)

    grid_min = np.min(work_vals) - 4 * effective_bw
    grid_max = np.max(work_vals) + 4 * effective_bw
    eval_grid = np.linspace(grid_min, grid_max, num_points)[:, np.newaxis]

    log_pdf = kde.score_samples(eval_grid)
    pdf_eval = np.exp(log_pdf)

    cdf_eval = np.cumsum(pdf_eval)
    cdf_eval = (cdf_eval - cdf_eval[0]) / (cdf_eval[-1] - cdf_eval[0])

    ranks = (stats.rankdata(work_vals, method="ordinal") - 1).astype(int)
    active_quantiles = (ranks + 0.5) / n_active

    smoothed_work = np.interp(
        active_quantiles, cdf_eval, eval_grid.squeeze()
    )
    return smoothed_work


def _match_active_moments(
    raw_active: np.ndarray,
    smoothed_active: np.ndarray,
    anchor_cutoff: float | None = None,
    enable_matching: bool = True,
) -> np.ndarray:
    """Perform direct moment matching to enforce exact active-domain mean and variance preservation.

    Parameters
    ----------
    raw_active : np.ndarray
        Original unsmoothed active values.
    smoothed_active : np.ndarray
        Quantile-mapped smoothed values.
    anchor_cutoff : float or None, default=None
        Lower bound truncation threshold to prevent clipping back below boundary.
    enable_matching : bool, default=True
        If False, returns smoothed_active directly without forcing raw mean/std.

    Returns
    -------
    np.ndarray
        Moment-adjusted smoothed array.
    """
    if not enable_matching or len(raw_active) == 0:
        return smoothed_active

    raw_mean = np.mean(raw_active)
    raw_std = np.std(raw_active, ddof=1)

    smooth_mean = np.mean(smoothed_active)
    smooth_std = np.std(smoothed_active, ddof=1)

    if smooth_std > 1e-8:
        adjusted = (smoothed_active - smooth_mean) * (
            raw_std / smooth_std
        ) + raw_mean
        if anchor_cutoff is not None:
            return np.clip(adjusted, anchor_cutoff, None)
        return adjusted
    return smoothed_active


def _reassemble_dataset(
    ds: xr.Dataset,
    var_name: str,
    raw_vals: np.ndarray,
    valid_mask: np.ndarray,
    valid_vals: np.ndarray,
    active_mask: np.ndarray,
    smoothed_active: np.ndarray,
) -> xr.Dataset:
    """Reconstruct target variable in original Dataset structure."""
    mapped_vals = np.copy(valid_vals)
    mapped_vals[active_mask] = smoothed_active
    mapped_vals[~active_mask] = valid_vals[~active_mask]

    out_vals = np.copy(raw_vals)
    out_vals[np.reshape(valid_mask, raw_vals.shape)] = mapped_vals.reshape(
        valid_vals.shape
    )

    da = ds[var_name]
    smoothed_da = da.copy(data=out_vals)
    smoothed_da.attrs = da.attrs.copy()
    smoothed_da.encoding = da.encoding.copy()

    ds_out = ds.copy()
    ds_out[var_name] = smoothed_da
    return ds_out


# =====================================================================
# KDE FIT FUNCTION
# =====================================================================


def smooth_temperature(
    ds: xr.Dataset,
    var_name: str,
    bw_factor: float = 1.0,
    seed: int = 902,
    match_moments: bool = True,
) -> xr.Dataset:
    """Smooth temperature variables (tas, tdps) over unconstrained continuous domains.

    Parameters
    ----------
    ds : xr.Dataset
        Input xarray Dataset.
    var_name : str
        Target temperature variable name.
    bw_factor : float, default=1.0
        Multiplier for Silverman's rule of thumb bandwidth.
    seed : int, default=902
        Random seed for dither generation.
    match_moments : bool, default=True
        If True, rescales smoothed values to match the raw mean and variance.
        Set to False during sensitivity analysis to allow quantile variance shifts to pass through.

    Returns
    -------
    xr.Dataset
        Dataset with smoothed temperature variable.
    """
    raw_vals = ds[var_name].values
    valid_mask = ~np.isnan(raw_vals.ravel())
    valid_vals = raw_vals.ravel()[valid_mask]

    if len(valid_vals) < 2:
        return ds.copy()

    # Dithering noise
    jittered_vals, effective_step = _compute_adaptive_jitter(valid_vals, seed)
    active_mask = np.ones_like(jittered_vals, dtype=bool)

    # Kernel Bandwidth Estimation
    std_dev = np.std(jittered_vals, ddof=1)
    iqr = stats.iqr(jittered_vals)
    scale = max(std_dev, iqr / 1.34) if iqr > 0 else std_dev
    silverman_bw = 0.9 * scale * (len(jittered_vals) ** (-0.2))
    min_bw = effective_step * 1.2
    effective_bw = max(silverman_bw, min_bw) * bw_factor

    # Kernel Density CDF Smoothing
    smoothed_work = _fit_kde_and_map_quantiles(
        work_vals=jittered_vals,
        effective_bw=effective_bw,
    )

    # Variance and Mean Alignment (Toggled via match_moments)
    smoothed_active = _match_active_moments(
        valid_vals,
        smoothed_work,
        anchor_cutoff=None,
        enable_matching=match_moments,
    )

    return _reassemble_dataset(
        ds,
        var_name,
        raw_vals,
        valid_mask,
        valid_vals,
        active_mask,
        smoothed_active,
    )

# =====================================================================
# KDE FIT VALIDATION + SENSITIVITY TESTS
# =====================================================================    

def verify_variance_preservation(
    ds_raw: xr.Dataset,
    ds_processed: xr.Dataset,
    var_name: str,
    threshold: float | None = None,
    rtol: float = 0.01,
) -> dict[str, float | bool]:
    """Calculate and compare active domain sample variance before and after smoothing.

    Parameters
    ----------
    ds_raw : xr.Dataset
        Original raw observational dataset.
    ds_processed : xr.Dataset
        Smoothed dataset.
    var_name : str
        Target variable name.
    threshold : float or None, default=None
        Censoring threshold boundary.
    rtol : float, default=0.01
        Relative variance tolerance threshold (1%).

    Returns
    -------
    dict
        Summary dictionary containing variance values, ratio, and verification flag.
    """
    raw_vals = ds_raw[var_name].values.ravel()
    proc_vals = ds_processed[var_name].values.ravel()

    mask = ~np.isnan(raw_vals) & ~np.isnan(proc_vals)
    raw_valid = raw_vals[mask]
    proc_valid = proc_vals[mask]

    if threshold is not None:
        active_mask = raw_valid > threshold
        raw_active = raw_valid[active_mask]
        proc_active = proc_valid[active_mask]
    else:
        raw_active = raw_valid
        proc_active = proc_valid

    var_raw = float(np.var(raw_active, ddof=1))
    var_proc = float(np.var(proc_active, ddof=1))

    var_ratio = var_proc / var_raw if var_raw > 0 else 1.0
    var_diff_pct = (var_proc - var_raw) / var_raw * 100.0 if var_raw > 0 else 0.0
    is_preserved = abs(var_proc - var_raw) / var_raw <= rtol if var_raw > 0 else True

    return {
        "var_raw": var_raw,
        "var_processed": var_proc,
        "variance_ratio": var_ratio,
        "pct_change": var_diff_pct,
        "is_preserved": is_preserved,
    }

## Step 3: Load Observations

In [ ]:
# Set 30-year reference analysis temporal bounds
ref_slice = ["1992-01-01T00:00:00", "2021-12-31T23:00:00"]

all_stns = gpd.read_file("hdp_wrf_locs_d03.gdb/")
stn_name = "KSAN"
stn_gdf = all_stns[all_stns["source-id"] == stn_name]

my_station = stn_gdf["era-id"]
my_network = stn_gdf["network"]

# Query Intake datastore
cat = intake.open_esm_datastore(
    "https://cadcat.s3.amazonaws.com/histwxstns/era-hdp-collection.json"
)
query = {"station_id": my_station.item()}
cat_subset = cat.search(**query)

dsets = cat_subset.to_dataset_dict(
    xarray_open_kwargs={"consolidated": False}, storage_options={"anon": True}
)
obs_ds = list(dsets.values())[0]

# Subset slice and target variables
obs_ds = obs_ds.sel(time=slice(ref_slice[0], ref_slice[1]))
obs_ds = obs_ds[
    [
        "tas",
        "tas_eraqc",
        "tdps",
        "tdps_eraqc",
    ]
]

obs_ds = obs_ds.convert_calendar("noleap")
obs_ds = obs_ds.compute()
obs_ds = obs_ds.squeeze()

# Remove and visualize observations with QAQC flag
obs_ds = apply_qaqc_and_plot_missing(obs_ds, stn_name=stn_name)

## Step 4: Inspect the discrete observational data

Show the reporting intervals and counts in the raw observational dataset.

In [ ]:
# Convert to degF
obs_ds['tas'] = convert_units_to(obs_ds['tas'], 'degF')
obs_ds['tdps'] = convert_units_to(obs_ds['tdps'], 'degF')
# Inspect unique discretization steps across variables
var_dfs = {}
decimals_list = [3] * len(obs_ds.data_vars)

for var, decimal in zip(obs_ds.data_vars, decimals_list):
    df = get_sorted_unique(obs_ds, var, decimal)
    var_dfs[var] = df

    print(f"\n=== Variable: {var} ===")
    display(df.T)

## Step 5: Execute continuous distribution smoothing & Perform Diagnostic validation — Density and bulk statistics

We evaluate the smoothed dataset across three validation checks:
1. **Continuous Kernel Density Estimates (KDE):** Verify discretization spikes are smoothed without distorting peak modes.
2. **Active Domain Variance Match:** Confirm preservation of sample variance and absolute mean bias.
3. **Long-Term Climate Means:** Verify annual mean and annual 95th percentile of the fitted data track the raw observations.

Start with temperature

In [ ]:
# Temperature Bandwidth Sensitivity Tests

# Generate smoothed data 
smoothed_tas_ds = smooth_temperature(
    obs_ds,
    var_name='tas',
    bw_factor=2.5,
    match_moments=True
)

# Print mean and variance to examine statistical consistency with respect to observations
print("--- Diagnostics: Temperature ---")
veri = verify_variance_preservation(obs_ds, smoothed_tas_ds, 'tas')
print(veri)
raw_mean = float(obs_ds['tas'].mean())
smooth_mean = float(smoothed_tas_ds['tas'].mean())
bias = smooth_mean - raw_mean
rel_bias = (bias / raw_mean) * 100

print(
    f"Temperature Raw Mean: {raw_mean:.4f} | Smoothed Mean: {smooth_mean:.4f} | "
    f"Absolute Bias: {bias:+.5f} | Relative Bias: {rel_bias:+.3f}%"
)

In [ ]:
# Prepare raw data series
raw_series = obs_ds['tas'].values.ravel()
raw_series = raw_series[~np.isnan(raw_series)]

# Figure setup
fig, ax = plt.subplots(figsize=(10, 5.5))

# 1. Plot Raw (Discretized) baseline line prominently
sns.kdeplot(
    data=raw_series,
    ax=ax,
    color="k",
    alpha=0.8,
    linestyle="--",
    linewidth=2.0,
    label="Raw observations",
    zorder=2,
)

# 2. Plot smoothed data
smoothed_series = smoothed_tas_ds['tas'].values.ravel()
smoothed_series = smoothed_series[~np.isnan(smoothed_series)]

sns.kdeplot(
    data=smoothed_series,
    ax=ax,
    color='tab:red',
    linestyle="-",
    linewidth=1.2,
    alpha=1,
    label="Smoothed observations",
    zorder=1,
)

# 3. Add a Zoomed Inset Panel to highlight the distribution peak
ax_inset = ax.inset_axes([0.65, 0.43, 0.38, 0.50])

sns.kdeplot(
    data=raw_series,
    ax=ax_inset,
    color="k",
    linestyle="--",
    linewidth=1.5,
    alpha=0.8,
)

sns.kdeplot(
    data=smoothed_series,
    ax=ax_inset,
    color="tab:red",
    linestyle="-",
    linewidth=1.2,
    alpha=1,
)

# Crop inset focused directly on peak region 
ax_inset.set_xlim(60, 67)
ax_inset.set_ylim(0.05, 0.062)
ax_inset.set_xlabel("")
ax_inset.set_ylabel("")
ax_inset.tick_params(labelsize=8)
ax.indicate_inset_zoom(ax_inset, edgecolor="#888888", alpha=0.7)

# Titles and formatting
ax.set_title(f"KDE Distribution of Temperature", fontsize=12, fontweight="bold")
ax.set_ylabel("Density")
ax.legend(
    frameon=True,
    facecolor="white",
    edgecolor="none",
    loc="upper left",
    fontsize=9,
)

plt.tight_layout()
plt.show()

In [ ]:
# Plot basic diagnostics: Annual mean and 95th percentile before and after fitting
fig, axes = plt.subplots(
    1, 2, figsize=(14, 3), squeeze=False
)
axes = axes.ravel()

raw_da = obs_ds['tas']
smooth_da = smoothed_tas_ds['tas']

# Annual mean
ax_trend = axes[0]
raw_annual = raw_da.resample(time="1YE").mean().squeeze()
smooth_annual = smooth_da.resample(time="1YE").mean().squeeze()


ax_trend.plot(
    raw_annual.time.dt.year,
    raw_annual.values,
    "k--",
    label="Raw",
    linewidth=3.5,
    alpha=0.6
)
ax_trend.plot(
    smooth_annual.time.dt.year,
    smooth_annual.values,
    color="tab:red",
    label="Smoothed",
    linewidth=1.5,
    alpha=1,
)


ax_trend.set_title("Temperature Annual Mean Trend Preservation")
ax_trend.set_ylabel(f"Mean ({raw_da.attrs.get('units', 'native units')})")
ax_trend.legend(loc="best")
ax_trend.grid(True, linestyle=":", alpha=0.6)

# Annual 95th percentile
ax_ext = axes[1]
raw_p95 = raw_da.resample(time="1YE").quantile(0.95).squeeze()
smooth_p95 = smooth_da.resample(time="1YE").quantile(0.95).squeeze()

ax_ext.plot(
    raw_p95.time.dt.year,
    raw_p95.values,
    "k--",
    label="Raw P95",
    linewidth=3.5,
    alpha=0.6
)
ax_ext.plot(
    smooth_p95.time.dt.year,
    smooth_p95.values,
    color="tab:red",
    label="Smoothed P95",
    alpha=1,
    linewidth=1.5
)

ax_ext.set_title(f"Temperature Annual 95th Percentile Trend")
ax_ext.set_ylabel("95th Percentile")
ax_ext.legend(loc="best")
ax_ext.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

Now let's do dewpoint

Preemptive note: The poor performance on the right tail is likely due to the absence of adaptive bandwidth, which is not yet implemented in `sklearn`.

In [ ]:
smoothed_tdps_ds = smooth_temperature(
    obs_ds,
    var_name='tdps',
    bw_factor=2.0,
    match_moments=True  # Linear correction back to original mean and variance
)

# Print mean and variance to examine statistical consistency with respect to observations
print("--- Diagnostics: Dewpoint ---")
veri = verify_variance_preservation(obs_ds, smoothed_tdps_ds, 'tas')
print(veri)
raw_mean = float(obs_ds['tdps'].mean())
smooth_mean = float(smoothed_tas_ds['tdps'].mean())
bias = smooth_mean - raw_mean
rel_bias = (bias / raw_mean) * 100

print(
    f"Dewpoint Raw Mean: {raw_mean:.4f} | Smoothed Mean: {smooth_mean:.4f} | "
    f"Absolute Bias: {bias:+.5f} | Relative Bias: {rel_bias:+.3f}%"
)

In [ ]:
# Prepare raw data series
raw_series = obs_ds['tdps'].values.ravel()
raw_series = raw_series[~np.isnan(raw_series)]

# Figure setup
fig, ax = plt.subplots(figsize=(10, 5.5))

# 1. Plot Raw (Discretized) baseline line prominently
sns.kdeplot(
    data=raw_series,
    ax=ax,
    color="k",
    alpha=0.8,
    linestyle="--",
    linewidth=2.0,
    label="Raw observations",
)

# 2. Plot smoothed data
smoothed_series = smoothed_tdps_ds['tdps'].values.ravel()
smoothed_series = smoothed_series[~np.isnan(smoothed_series)]

sns.kdeplot(
    data=smoothed_series,
    ax=ax,
    color='tab:red',
    linestyle="-",
    linewidth=1.2,
    alpha=0.85,
    label="Smoothed observations",
)

# 3. Add a Zoomed Inset Panel to highlight the distribution peak
ax_inset = ax.inset_axes([0.3, 0.43, 0.25, 0.50])

sns.kdeplot(
    data=raw_series,
    ax=ax_inset,
    color="k",
    linestyle="--",
    linewidth=1.5,
    alpha=0.8,
)

sns.kdeplot(
    data=smoothed_series,
    ax=ax_inset,
    color='tab:red',
    linestyle="-",
    linewidth=1.2,
    alpha=0.85,
)

# Crop inset focused on tail to show unfortunate widening
ax_inset.set_xlim(68, 78)
ax_inset.set_ylim(0.0, 0.002)
ax_inset.set_xlabel("")
ax_inset.set_ylabel("")
ax_inset.tick_params(labelsize=8)
ax.indicate_inset_zoom(ax_inset, edgecolor="#888888", alpha=0.7)

# Titles and formatting
ax.set_title("KDE Distribution of Dewpoint", fontsize=12, fontweight="bold")
ax.set_ylabel("Density")
ax.legend(
    frameon=True,
    facecolor="white",
    edgecolor="none",
    loc="upper left",
    fontsize=9,
)

plt.tight_layout()
plt.show()

In [ ]:
# Plot basic diagnostics: Annual mean and 95th percentile before and after fitting
fig, axes = plt.subplots(
    1, 2, figsize=(14, 3), squeeze=False
)
axes = axes.ravel()

raw_da = obs_ds['tdps']
smooth_da = smoothed_tas_ds['tdps']

# Annual mean
ax_trend = axes[0]
raw_annual = raw_da.resample(time="1YE").mean().squeeze()
smooth_annual = smooth_da.resample(time="1YE").mean().squeeze()


ax_trend.plot(
    raw_annual.time.dt.year,
    raw_annual.values,
    "k--",
    label="Raw",
    linewidth=3.5,
    alpha=0.6
)
ax_trend.plot(
    smooth_annual.time.dt.year,
    smooth_annual.values,
    color="tab:red",
    label="Smoothed",
    linewidth=1.5,
    alpha=1,
)


ax_trend.set_title("Dewpoint Annual Mean Trend Preservation")
ax_trend.set_ylabel(f"Mean ({raw_da.attrs.get('units', 'native units')})")
ax_trend.legend(loc="best")
ax_trend.grid(True, linestyle=":", alpha=0.6)

# Annual 95th percentile
ax_ext = axes[1]
raw_p95 = raw_da.resample(time="1YE").quantile(0.95).squeeze()
smooth_p95 = smooth_da.resample(time="1YE").quantile(0.95).squeeze()

ax_ext.plot(
    raw_p95.time.dt.year,
    raw_p95.values,
    "k--",
    label="Raw P95",
    linewidth=3.5,
    alpha=0.6
)
ax_ext.plot(
    smooth_p95.time.dt.year,
    smooth_p95.values,
    color="tab:red",
    label="Smoothed P95",
    alpha=1,
    linewidth=1.5
)

ax_ext.set_title(f"Dewpoint Annual 95th Percentile Trend")
ax_ext.set_ylabel("95th Percentile")
ax_ext.legend(loc="best")
ax_ext.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

## Step 6: Save the dithered observations to make the next notebooks less overwhelming

In [ ]:
# At least this one
smoothed_tas_ds.to_zarr("ksan_smoothed_tas", mode="w")

# If you want to change step2 to dewpoint
# smoothed_tdps_ds.to_zarr("ksan_smoothed_tdps", mode="w")